In [1]:
%pip install nibabel

Looking in indexes: http://cor-notebook-dev-wheel-cache.projects:8081/simple, https://download.pytorch.org/whl/cu130
Looking in links: /var/cache/pip/wheels-local, /var/cache/pip/wheels
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import random
import sys
import time
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from sklearn.metrics import average_precision_score, roc_auc_score
from torch.optim import AdamW
from torch.utils.data import DataLoader
from tqdm import tqdm


In [3]:
TRAINING_LOOP_DIR = Path(".").resolve()
REPO_DIR = TRAINING_LOOP_DIR.parent
sys.path.insert(0, str(TRAINING_LOOP_DIR.resolve()))

DATASET_DIR = REPO_DIR.parent / "dataset"
TRAIN_DIR = DATASET_DIR / "Dataset_train"
VAL_DIR = DATASET_DIR / "Dataset_validation"
TRAIN_CSV = REPO_DIR / "tags" / "train.csv"
VAL_CSV = REPO_DIR / "tags" / "validation.csv"
OUTPUT_DIR = REPO_DIR / "output" / "history_3d_residual_cnn"


In [4]:
label_columns = [
        "ICH" ]

target_size = 128


In [9]:
from dataset_3d_binary import CTVolumeDataset

train_dataset = CTVolumeDataset(
    table_path=TRAIN_CSV,
    images_dir=TRAIN_DIR,
    label_columns=label_columns,
    target_size=target_size,
)

val_dataset = CTVolumeDataset(
    table_path=VAL_CSV,
    images_dir=VAL_DIR,
    label_columns=label_columns,
    target_size=target_size,
)



In [11]:
class ConvBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout_p=0.0):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False),
            nn.InstanceNorm3d(out_channels),
            nn.ReLU(inplace=True),
            nn.Dropout3d(p=dropout_p) if dropout_p > 0 else nn.Identity(),
        )

    def forward(self, x):
        return self.block(x)


class ResidualBlock3D(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1, dropout_p=0.0):
        super().__init__()

        self.conv1 = nn.Conv3d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1, bias=False)
        self.norm1 = nn.InstanceNorm3d(out_channels)
        self.conv2 = nn.Conv3d(out_channels, out_channels, kernel_size=3, stride=1, padding=1, bias=False)
        self.norm2 = nn.InstanceNorm3d(out_channels)
        self.dropout = nn.Dropout3d(p=dropout_p) if dropout_p > 0 else nn.Identity()

        if stride != 1 or in_channels != out_channels:
            self.skip = nn.Sequential(
                nn.Conv3d(in_channels, out_channels, kernel_size=1, stride=stride, bias=False),
                nn.InstanceNorm3d(out_channels),
            )
        else:
            self.skip = nn.Identity()

    def forward(self, x):
        identity = self.skip(x)

        out = F.relu(self.norm1(self.conv1(x)), inplace=True)
        out = self.dropout(out)
        out = self.norm2(self.conv2(out))
        out = F.relu(out + identity, inplace=True)
        return out


class Residual3DClassifier(nn.Module):
    def __init__(self, in_channels=1, num_classes=1):
        super().__init__()

        self.encoder = nn.Sequential(
            ConvBlock3D(in_channels, 16, stride=2, dropout_p=0.1),
            ResidualBlock3D(16, 16, stride=1, dropout_p=0.1),
            ResidualBlock3D(16, 32, stride=2, dropout_p=0.1),
            ResidualBlock3D(32, 32, stride=1, dropout_p=0.1),
            ResidualBlock3D(32, 64, stride=2, dropout_p=0.1),
            ResidualBlock3D(64, 64, stride=1, dropout_p=0.1),
            ResidualBlock3D(64, 128, stride=2, dropout_p=0.1),
            ResidualBlock3D(128, 128, stride=1, dropout_p=0.1),
            ResidualBlock3D(128, 256, stride=2, dropout_p=0.1),
            ResidualBlock3D(256, 256, stride=1, dropout_p=0.1),
        )

        self.head = nn.Sequential(
            nn.AdaptiveAvgPool3d(1),
            nn.Flatten(),
            nn.Linear(256, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.encoder(x)
        x = self.head(x)
        return x


In [12]:
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

batch_size = 8
num_epochs = 50
print_every_i = 10  
learning_rate = 1e-4
num_workers = 4
loss_log_every_i = 10
prefetch_factor = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
train_loader_generator = torch.Generator().manual_seed(seed)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    prefetch_factor = prefetch_factor,
    pin_memory=(device.type == "cuda"),
    generator=train_loader_generator,
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    prefetch_factor = prefetch_factor,
    pin_memory=(device.type == "cuda"),
)
model = Residual3DClassifier(in_channels=1, num_classes=len(label_columns)).to(device)

criterion = nn.BCEWithLogitsLoss()

start_epoch = 1

optimizer = AdamW(model.parameters(), lr=learning_rate)


OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

epoch_metrics_path = OUTPUT_DIR / "epoch_metrics.csv"
iter_losses_path = OUTPUT_DIR / "iter_losses_every_10.csv"
def calculate_epoch_metrics(y_true, y_score):
    y_true = np.asarray(y_true)
    y_score = np.asarray(y_score)
    return {
        "roc_auc": round(roc_auc_score(y_true, y_score), 5),
        "pr_auc": round(average_precision_score(y_true, y_score), 5),
    }


for epoch in range(start_epoch, num_epochs + 1):
    epoch_start_time = time.time()
    model.train()
    epoch_loss_sum = 0.0

    iter_losses_current_epoch = []
    train_targets = []
    train_scores = []

    for i, (volumes, labels) in enumerate(tqdm(train_loader, desc=f"Epoch {epoch}/{num_epochs}", leave=False), start=1):
        volumes = volumes.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)
        logits = model(volumes)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()

        iter_loss = round(loss.item(), 5)
        epoch_loss_sum += loss.item()
        train_targets.extend(labels.detach().cpu().numpy().reshape(-1).tolist())
        train_scores.extend(torch.sigmoid(logits).detach().cpu().numpy().reshape(-1).tolist())

        if i % loss_log_every_i == 0:
            iter_losses_current_epoch.append(
                {
                    "epoch": epoch,
                    "split": "train",
                    "iteration": i,
                    "loss": iter_loss,
                }
            )


    epoch_loss = round(epoch_loss_sum / max(len(train_loader), 1), 5)

    model.eval()
    val_loss_sum = 0.0
    val_targets = []
    val_scores = []

    with torch.no_grad():
        for i, (volumes, labels) in enumerate(val_loader, start=1):
            volumes = volumes.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)

            logits = model(volumes)
            loss = criterion(logits, labels)
            val_loss_sum += loss.item()
            val_targets.extend(labels.detach().cpu().numpy().reshape(-1).tolist())
            val_scores.extend(torch.sigmoid(logits).detach().cpu().numpy().reshape(-1).tolist())

    val_loss = round(val_loss_sum / max(len(val_loader), 1), 5)
    train_epoch_metrics = calculate_epoch_metrics(train_targets, train_scores)
    val_epoch_metrics = calculate_epoch_metrics(val_targets, val_scores)
    epoch_time_min = round((time.time() - epoch_start_time) / 60.0, 2)

    epoch_row = pd.DataFrame(
        [{
            "epoch": epoch,
            "train_loss": epoch_loss,
            "val_loss": val_loss,
            "train_roc_auc": train_epoch_metrics["roc_auc"],
            "train_pr_auc": train_epoch_metrics["pr_auc"],
            "val_roc_auc": val_epoch_metrics["roc_auc"],
            "val_pr_auc": val_epoch_metrics["pr_auc"],
            "epoch_time_min": epoch_time_min,
        }]
    )

    epoch_row.to_csv(
        epoch_metrics_path,
        mode="a",
        header=not epoch_metrics_path.exists(),
        index=False
    )

    if iter_losses_current_epoch:
        pd.DataFrame(iter_losses_current_epoch).to_csv(
            iter_losses_path,
            mode="a",
            header=not iter_losses_path.exists(),
            index=False
        )

    
    torch.save(
        {
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
        },
        OUTPUT_DIR / f"checkpoint_epoch_{epoch}.pt"
    )



Using device: cuda


KeyboardInterrupt: 